In [1]:
!pip install -q -U trl peft bitsandbytes accelerate datasets

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import glob
import inspect
import json
import re
import pandas as pd
import torch
from concurrent.futures import ThreadPoolExecutor
from datasets import Dataset
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer

# Tìm tự động các file dữ liệu và SFT adapter đã lưu
grpo_files = glob.glob("/kaggle/input/**/D_GRPO.parquet", recursive=True)
test_files = glob.glob("/kaggle/input/**/test_9600.parquet", recursive=True)
adapter_files = glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)

assert len(grpo_files) > 0 and len(test_files) > 0, "Thiếu dataset read-splits-v1 hoặc read-distilled-v1!"
assert len(adapter_files) > 0, "Chưa gắn dataset student-sft-adapter-v1 vào Input!"

SFT_ADAPTER_DIR = os.path.dirname(adapter_files[0])
df_grpo = pd.read_parquet(grpo_files[0])
df_test = pd.read_parquet(test_files[0])

print(f"Đã nạp SFT adapter tại: {SFT_ADAPTER_DIR}", flush=True)
print(f"D_GRPO pool: {len(df_grpo):,} | Test: {len(df_test):,}", flush=True)

SYSTEM_PROMPT = (
    "You are an expert in textual forensics. Decide whether the given text is AI-generated or human-written. "
    "Reason step by step considering style, lexical diversity, sentence rhythm, structural symmetry, and concrete specificity. "
    "Use transition words such as 'First', 'Second', 'Moreover' and end your reasoning with your conclusion. "
    "Return STRICT JSON ONLY with exact keys: "
    '{"rationale": "3-5 concise text-specific sentences", "verdict": "AI" or "HUMAN"}.'
)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
Đã nạp SFT adapter tại: /kaggle/input/datasets/huynhthanhtra/student-sft-adapter-v1/student_sft_adapter
D_GRPO pool: 41,049 | Test: 9,600


In [2]:
# 1. Chuẩn bị tập huấn luyện GRPO cân bằng 330 mẫu (15 mẫu/nhãn/miền)
df_grpo_sub = (
    df_grpo.groupby(["domain", "label"], group_keys=False)
    .apply(lambda g: g.sample(n=min(15, len(g)), random_state=42))
    .sample(frac=1.0, random_state=42)
    .reset_index(drop=True)
)

tokenizer_left = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
if tokenizer_left.pad_token is None:
    tokenizer_left.pad_token = tokenizer_left.eos_token

def format_grpo_prompt(row):
    text_clip = " ".join(str(row["text"]).split()[:600])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Analyze this text:\n\n{text_clip}"},
    ]
    return {
        "prompt": tokenizer_left.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
        "ground_truth": "AI" if str(row["label"]).strip().upper() == "AI" else "HUMAN",
    }

ds_grpo = Dataset.from_pandas(df_grpo_sub).map(format_grpo_prompt, remove_columns=df_grpo_sub.columns.tolist())

def parse_prediction(text):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None, None
    try:
        obj = json.loads(match.group(0))
        rat = str(obj.get("rationale", "")).strip()
        ver = str(obj.get("verdict", "")).strip().upper()
        if ver in {"AI", "HUMAN"} and len(rat) >= 30:
            return rat, ver
    except Exception:
        pass
    return None, None

def format_reward_func(completions, **kwargs):
    return [0.5 if parse_prediction(c)[1] is not None else -0.5 for c in completions]

def accuracy_reward_func(completions, ground_truth, **kwargs):
    rewards = []
    for comp, gt in zip(completions, ground_truth):
        _, ver = parse_prediction(comp)
        rewards.append(1.5 if ver == gt else 0.0)
    return rewards

# 2. Nạp SFT Adapter lên GPU 0 và chạy GRPO bản nhanh (~25 phút)
sft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map={"": 0},
)
model_grpo = PeftModel.from_pretrained(sft_base, SFT_ADAPTER_DIR, is_trainable=True)

grpo_out_dir = "/kaggle/working/student_grpo_adapter"
valid_grpo_params = inspect.signature(GRPOConfig.__init__).parameters
grpo_kwargs = {
    "output_dir": grpo_out_dir,
    "learning_rate": 5e-6,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "num_generations": 2,
    "num_train_epochs": 1,
    "fp16": False,
    "bf16": False,
    "logging_steps": 10,
    "save_strategy": "no",
    "report_to": "none",
}
if "max_completion_length" in valid_grpo_params:
    grpo_kwargs["max_completion_length"] = 200
elif "max_new_tokens" in valid_grpo_params:
    grpo_kwargs["max_new_tokens"] = 200

grpo_trainer = GRPOTrainer(
    model=model_grpo,
    reward_funcs=[format_reward_func, accuracy_reward_func],
    args=GRPOConfig(**grpo_kwargs),
    train_dataset=ds_grpo,
    processing_class=tokenizer_left,
)

grpo_trainer.train()
grpo_trainer.model.save_pretrained(grpo_out_dir)
tokenizer_left.save_pretrained(grpo_out_dir)
print("Đã lưu xong mô hình chính (Student-GRPO) tại:", grpo_out_dir, flush=True)

del grpo_trainer, model_grpo, sft_base
torch.cuda.empty_cache()

# 3. CHẤM ĐIỂM TẬP TEST PHÂN TẦNG 2,200 MẪU (100/NHÃN/MIỀN) SONG SONG 2 GPU (~15 PHÚT)
df_test_sub = (
    df_test.groupby(["domain", "label"], group_keys=False)
    .apply(lambda g: g.sample(n=min(100, len(g)), random_state=42))
    .reset_index(drop=True)
)
print(f"Đánh giá nhanh trên tập con phân tầng cân bằng 11 miền: {len(df_test_sub):,} mẫu", flush=True)

sub_col = "sub_source" if "sub_source" in df_test_sub.columns else ("source" if "source" in df_test_sub.columns else None)
src_ds_col = "source_dataset" if "source_dataset" in df_test_sub.columns else None

def eval_worker(gpu_id, df_shard):
    device = f"cuda:{gpu_id}"
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        dtype=torch.float16,
        device_map={"": gpu_id},
    )
    model = PeftModel.from_pretrained(base, grpo_out_dir).eval()
    shard_res = []
    BATCH_EVAL = 64

    for start in tqdm(range(0, len(df_shard), BATCH_EVAL), desc=f"GPU {gpu_id} Test Eval", position=gpu_id):
        batch_df = df_shard.iloc[start : start + BATCH_EVAL]
        prompts = []
        for _, row in batch_df.iterrows():
            text_clip = " ".join(str(row["text"]).split()[:600])
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Analyze this text:\n\n{text_clip}"},
            ]
            prompts.append(tokenizer_left.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

        inputs = tokenizer_left(prompts, return_tensors="pt", padding=True, truncation=True, max_length=950).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=tokenizer_left.pad_token_id,
            )

        gen_tokens = outputs[:, inputs["input_ids"].shape[1] :]
        decoded = tokenizer_left.batch_decode(gen_tokens, skip_special_tokens=True)

        for (_, row), raw_out in zip(batch_df.iterrows(), decoded):
            rat, ver = parse_prediction(raw_out)
            shard_res.append({
                "sample_id": int(row["sample_id"]),
                "domain": row["domain"],
                "sub_source": row[sub_col] if sub_col else "",
                "source_dataset": row[src_ds_col] if src_ds_col else "",
                "label": row["label"],
                "raw_response": raw_out.strip(),
                "rationale": rat if rat is not None else "",
                "verdict": ver if ver is not None else "INVALID",
                "pred_label": "AI" if ver == "AI" else ("Human" if ver == "HUMAN" else "INVALID"),
            })
    return shard_res

n_gpus = torch.cuda.device_count()
if n_gpus >= 2:
    mid = len(df_test_sub) // 2
    with ThreadPoolExecutor(max_workers=2) as ex:
        f0 = ex.submit(eval_worker, 0, df_test_sub.iloc[:mid])
        f1 = ex.submit(eval_worker, 1, df_test_sub.iloc[mid:])
        results = f0.result() + f1.result()
else:
    results = eval_worker(0, df_test_sub)

eval_out_path = "/kaggle/working/student_grpo_test_predictions.parquet"
df_eval = pd.DataFrame(results)
df_eval.to_parquet(eval_out_path, index=False)

valid_df = df_eval[df_eval["pred_label"] != "INVALID"]
acc_all = (df_eval["pred_label"] == df_eval["label"]).mean()
acc_valid = (valid_df["pred_label"] == valid_df["label"]).mean() if len(valid_df) > 0 else 0.0
fer = 1.0 - (len(valid_df) / len(df_eval))

print("\n=== KẾT QUẢ MÔ HÌNH CHÍNH (STUDENT SFT+GRPO) TRÊN TEST PHÂN TẦNG 2,200 ===", flush=True)
print(f"Accuracy toàn bộ (tính cả lỗi format) : {acc_all:.4f}", flush=True)
print(f"Accuracy trên các mẫu hợp lệ         : {acc_valid:.4f}", flush=True)
print(f"Format Error Rate (FER)              : {fer:.4f}", flush=True)

/tmp/ipykernel_23/145725693.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(15, len(g)), random_state=42))


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (8 outputs left out)

/tmp/ipykernel_23/145725693.py:98: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(100, len(g)), random_state=42))


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

GPU 0 Test Eval:   0%|          | 0/35 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 3.97 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.93 GiB is free. Including non-PyTorch memory, this process has 12.63 GiB memory in use. Of the allocated memory 9.39 GiB is allocated by PyTorch, and 3.10 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)